# Flow Shop híbrido FH4 con algoritmo genético

**FH4, ((P4), (Q20), (P9), (P8)) | ls, trans, am | ΣTj**

Este cuaderno funciona en **Google Colab** o en **Jupyter** local. Ejecute las celdas en orden.

## 1. Preparar el entorno
En Colab, clona el repositorio. En Jupyter local, abra este cuaderno desde la carpeta del proyecto y omita la clonación.

In [ ]:
import os, sys
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB and not os.path.exists('Flow-Shop-hibrido'):
    !git clone https://github.com/luisaghs/Flow-Shop-hibrido.git
if EN_COLAB:
    %cd Flow-Shop-hibrido
!pip install -q -r requirements.txt

## 2. Subir sus datos

1. Descargue la plantilla `datos/plantilla_datos.xlsx` (o ejecute la celda siguiente para generarla).
2. Llénela con sus datos (hojas **Trabajos**, **Tiempos**, **Maquinas**, **Transporte**; tiempos en **horas**).
3. Ejecute la celda de carga: en Colab aparece un botón **"Elegir archivos"**; en Jupyter local escriba la ruta en `RUTA_DATOS`.

Si no sube nada, se usa la plantilla de ejemplo.

In [ ]:
!python generar_plantilla.py
RUTA_DATOS = 'datos/plantilla_datos.xlsx'   # <-- en Jupyter local cambie aquí la ruta de su archivo

In [ ]:
if EN_COLAB:
    from google.colab import files
    subidos = files.upload()          # botón para seleccionar su .xlsx
    if subidos:
        nombre = next(iter(subidos))
        os.makedirs('datos', exist_ok=True)
        RUTA_DATOS = os.path.join('datos', nombre)
        with open(RUTA_DATOS, 'wb') as f:
            f.write(subidos[nombre])
print('Datos a utilizar:', RUTA_DATOS)

## 3. Cargar y revisar los datos

In [ ]:
from hfs_ga import cargar_datos, AlgoritmoGenetico, ParametrosAG, programa_a_tablas, guardar_todo
inst = cargar_datos(RUTA_DATOS, max_sublotes_defecto=3)
print(inst.resumen())

## 4. Parámetros del algoritmo genético y ejecución

In [ ]:
params = ParametrosAG(
    tam_poblacion=80,
    generaciones=300,
    prob_cruce=0.9,
    prob_mutacion=0.3,
    elitismo=2,
    tam_torneo=3,
    max_sin_mejora=100,
    tam_min_sublote=1,
    semilla=42,
)
res = AlgoritmoGenetico(inst, params).ejecutar()
print(f'ΣTj = {res.programa.tardanza_total:.3f} h | Cmax = {res.programa.makespan:.3f} h')

## 5. Resultados

In [ ]:
ops, trab = programa_a_tablas(res.programa, inst)
display(trab)
display(ops.head(30))

In [ ]:
rutas = guardar_todo(res, inst, params, 'resultados')
from IPython.display import Image, display
display(Image(rutas['convergencia']))
display(Image(rutas['gantt']))

In [ ]:
# Descargar el Excel de resultados (Colab)
if EN_COLAB:
    files.download(rutas['excel'])